In [16]:
import pandas as pd

df1 = pd.read_parquet("../../data/out/single_token_entropy/mmlu_mistral_24b.parquet")
df2 = pd.read_parquet("../../data/out/single_token_entropy/mmlu_qwen_32b.parquet")

In [17]:
df1.columns

Index(['src', 'answer', 'options', 'category', 'question', 'cot_content',
       'question_id', 'answer_index', 'total_tokens', 'meta_cluster',
       'base_cluster', 'model_answer', 'model_answer_correct',
       'entropy_value'],
      dtype='str')

In [18]:
missing = ~df1["question_id"].isin(df2["question_id"])
assert not missing.any(), f"{missing.sum()} rows in df1 not found in df2"

model1 = "mistral_24b"
model2 = "qwen_32b"

df1_renamed = df1.rename(
    columns={
        "model_answer": f"model_answer_{model1}",
        "model_answer_correct": f"model_answer_correct_{model1}",
        "entropy_value": f"entropy_value_{model1}",
    }
)

df2_subset = df2[["question_id", "model_answer", "model_answer_correct", "entropy_value"]].rename(
    columns={
        "model_answer": f"model_answer_{model2}",
        "model_answer_correct": f"model_answer_correct_{model2}",
        "entropy_value": f"entropy_value_{model2}",
    }
)

merged = df1_renamed.merge(df2_subset, on="question_id")
assert len(merged) == len(df1), "merge changed row count"

merged["ensemble_mean_entropy"] = (
    merged[f"entropy_value_{model1}"] + merged[f"entropy_value_{model2}"]
) / 2
merged

,src,answer,options,category,question,cot_content,question_id,answer_index,total_tokens,meta_cluster,base_cluster,model_answer_mistral_24b,model_answer_correct_mistral_24b,entropy_value_mistral_24b,model_answer_qwen_32b,model_answer_correct_qwen_32b,entropy_value_qwen_32b,ensemble_mean_entropy
0,ori_mmlu-jurisprudence,C,['There is no distinction between the two form...,law,Which of the following criticisms of Llewellyn...,NaN,1286,2,81,Legal Interpretation,Legal Theory Interpretations,c,True,0.776255,c,True,0.031885,0.404070
1,ori_mmlu-international_law,E,"['Article 19', 'Article 11', 'Article 12', 'Ar...",law,Which of the following articles are not qualif...,NaN,1293,4,38,Legal Interpretation,Constitutional Law,d,False,2.166177,b,False,1.149851,1.658014
2,ori_mmlu-management,D,"['Work delegation', 'Workload balancing', 'Wor...",business,As what is ensuring that one individual does n...,NaN,83,3,49,Economics & Finance MCQs,Business & Marketing Queries,b,False,1.039556,b,False,0.000077,0.519816
3,stemez-Business,J,"['$308.25', '$142.75', '$199.99', '$225.85', '...",business,Margaret Denault recently rented a truck to dr...,NaN,94,9,118,Economics & Finance MCQs,Business Finance Questions,d,False,2.773144,i,False,0.037588,1.405366
4,stemez-Business,I,"['$60,000', '$43,200', '$1,794', '$25,000', '$...",business,The tax rate in the town of Centerville is 11(...,NaN,104,8,102,Economics & Finance MCQs,Business Finance Questions,a,False,2.643690,i,True,0.062827,1.353258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12027,ori_mmlu-high_school_macroeconomics,F,['Higher interest rates that result from borro...,economics,"The ""crowding-out"" effect refers to which of t...",NaN,7681,5,150,Economics & Finance MCQs,Economic Concepts & Policies,f,True,0.788765,f,True,0.002121,0.395443
12028,ori_mmlu-high_school_macroeconomics,A,['Lower reserve requirements; lower the discou...,economics,Which of the following lists contains only Fed...,NaN,7683,0,124,Economics & Finance MCQs,Economic Concepts & Policies,a,True,0.482885,a,True,0.000326,0.241606
12029,ori_mmlu-high_school_macroeconomics,I,['The productivity of labor in country X is 75...,economics,Output in country X is 30000 units and there a...,NaN,7684,8,206,Economics & Finance MCQs,Economic Concepts & Policies,c,False,2.168652,b,False,0.165412,1.167032
12030,ori_mmlu-high_school_macroeconomics,B,"['an increase in net exports', 'a decrease in ...",economics,A use of easy money (expansionary) policy by t...,NaN,7685,1,58,Economics & Finance MCQs,Economic Concepts & Policies,b,True,0.819539,a,False,0.010439,0.414989


In [19]:
nan_count = merged["ensemble_mean_entropy"].isna().sum()
assert nan_count == 0, f"{nan_count} NaNs in ensemble_mean_entropy"
print("No NaNs")

No NaNs


In [20]:
merged.to_parquet("../../data/out/single_token_entropy/mmlu_24_32b_ensemble.parquet", index=False)